# Direct FLOW Evaluation (No SteeringPipeline)

This notebook loads a saved FLOW vector payload (`vector.pt` + `metadata.pt`) and evaluates transport quality directly in activation space.

Steps:
1. Load model + dataset via config and data registry
2. Collect last-token activations for target and contrast texts
3. Run saved FLOW model on contrast activations
4. Compare against target activations using FD / FD_1d

In [5]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformer_lens import HookedTransformer

os.environ.setdefault("HF_HOME", "/data/caotue/hf_cache")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/data/caotue/hf_cache/hub")
os.environ.setdefault("TRANSFORMERS_CACHE", "/data/caotue/hf_cache/transformers")
os.environ.setdefault("HF_DATASETS_CACHE", "/data/caotue/hf_cache/datasets")

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "Steering" / "post_process").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Could not locate repository root")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from Steering.config import PipelineConfig
from Steering.data.loader import DataLoader
from Steering.flow_utils import FlowMLP, denormalize, normalize, project_to_basis, solve_flow, unproject_from_basis
from Steering.post_process.fd_utils import compute_fd, compute_fd_1d
from Steering.utils import get_hook_name

config_path = repo_root / "Configs" / "Eval" / "FLOW" / "flowsteer_refusal_response.json"
vector_dir = repo_root / "Vector" / "Flow" / "gemma_refusal_response_subspace_l4_e200"
data_registry = "refusal_cast_responses"
n_samples = None
batch_size = 8
max_length = 1024
flow_steps = 20
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)

print("config_path:", config_path)
print("vector_dir:", vector_dir)
print("data_registry:", data_registry)
print("device:", device)

config_path: /home/caotue/SAESteeringBench/Configs/Eval/FLOW/flowsteer_refusal_response.json
vector_dir: /home/caotue/SAESteeringBench/Vector/Flow/gemma_refusal_response_subspace_l4_e200
data_registry: refusal_cast_responses
device: cuda


In [6]:
def _encode_batch(model, texts, max_length=1024):
    tokens = model.tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )
    model_device = next(model.parameters()).device
    return {k: v.to(model_device) for k, v in tokens.items()}


def collect_last_token_activations(model, texts, hook_name, batch_size=8, max_length=1024):
    all_vecs = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start : start + batch_size]
            tokens = _encode_batch(model, batch, max_length=max_length)
            _, cache = model.run_with_cache(
                tokens["input_ids"],
                names_filter=[hook_name],
                return_type=None,
            )
            acts = cache[hook_name].detach()
            last_idx = tokens["attention_mask"].sum(dim=1) - 1
            bidx = torch.arange(acts.shape[0], device=acts.device)
            vecs = acts[bidx, last_idx.to(acts.device), :].to(torch.float32).cpu()
            all_vecs.append(vecs)
    return torch.cat(all_vecs, dim=0)


def _move_nested_tensors_to(payload, device):
    out = {}
    for k, v in payload.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(device)
        elif isinstance(v, dict):
            out[k] = _move_nested_tensors_to(v, device)
        else:
            out[k] = v
    return out


def build_flow_inference_fn(payload, flow_steps):
    payload = _move_nested_tensors_to(payload, device)

    flow_model = FlowMLP(
        dim=int(payload["dim"]),
        hidden_dim=int(payload["hidden_dim"]),
        n_layers=int(payload["n_layers"]),
    ).to(device)
    flow_model.load_state_dict(payload["state_dict"])
    flow_model.eval()

    # Current payload keys (with backward-compatible fallbacks).
    basis = payload.get("flow_basis", payload.get("flow_subspace"))
    if isinstance(basis, torch.Tensor):
        basis = basis.to(device=device, dtype=torch.float32)
    else:
        basis = None

    mean = payload.get("flow_basis_mean", payload.get("flow_subspace_mean"))
    if isinstance(mean, torch.Tensor):
        mean = mean.to(device=device, dtype=torch.float32)
    else:
        mean = None

    basis_inv = payload.get("flow_basis_inv")
    if isinstance(basis_inv, torch.Tensor):
        basis_inv = basis_inv.to(device=device, dtype=torch.float32)
    else:
        basis_inv = None

    train_space = str(payload.get("flow_train_space", "full")).strip().lower()
    reduced_spaces = {"pca_diff", "pca_stack", "lda", "subspace"}

    def _restore_from_basis(coords):
        if basis is None:
            return coords
        if basis_inv is not None:
            y = coords @ basis_inv.T
            if mean is not None:
                y = y + mean
            return y
        return unproject_from_basis(coords, basis, mean)

    def transport(x):
        x = x.to(device=device, dtype=torch.float32)
        if basis is not None and train_space in reduced_spaces:
            x_coords = project_to_basis(x, basis, mean)
        else:
            x_coords = x

        x_norm = normalize(x_coords, payload["source_stats"])
        with torch.no_grad():
            solved = solve_flow(flow_model, x_norm, steps=flow_steps)
        y_coords = denormalize(solved, payload["target_stats"])

        if basis is not None and train_space in reduced_spaces:
            y = _restore_from_basis(y_coords)
        else:
            y = y_coords
        return y

    return transport

In [7]:
config = PipelineConfig.load(config_path)
model_name = config.model.name
model_dtype = config.model.get_dtype()

# Keep notebook defaults but prefer config values when present.
flow_steps = int(getattr(config.steer, "flow_steps", flow_steps))
if getattr(config, "load_vector", None):
    _cfg_vec = Path(config.load_vector)
    if not _cfg_vec.is_absolute():
        _cfg_vec = (repo_root / _cfg_vec).resolve()
    vector_dir = _cfg_vec

layer_cfg = config.extractor.layer
layer = int(layer_cfg[0] if isinstance(layer_cfg, list) else layer_cfg)
hook_cfg = config.extractor.hook_point
hook_point = hook_cfg[0] if isinstance(hook_cfg, list) else hook_cfg
hook_name = get_hook_name(layer, hook_point)

print("Loading model:", model_name)
model = HookedTransformer.from_pretrained(model_name, dtype=model_dtype, device=device)
if getattr(model.tokenizer, "pad_token", None) is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

loader = DataLoader()
dataset = loader.load(
    data_registry,
    n_samples=n_samples,
    format=True,
    apply_chat_template=bool(config.extractor.apply_chat_template),
    tokenizer=model.tokenizer,
)
dataset_cfg = loader.get_config(data_registry)

target_key = dataset_cfg.target_key
contrast_key = dataset_cfg.contrast_key
if not contrast_key:
    raise ValueError(f"Dataset '{data_registry}' has no contrast_key; choose a contrastive registry")

target_texts = [row[target_key] for row in dataset if row.get(target_key)]
contrast_texts = [row[contrast_key] for row in dataset if row.get(contrast_key)]
n = min(len(target_texts), len(contrast_texts))
target_texts = target_texts[:n]
contrast_texts = contrast_texts[:n]

print("hook_name:", hook_name)
print("layer:", layer)
print("samples:", n)
print("flow_steps:", flow_steps)
print("vector_dir:", vector_dir)

Loading model: google/gemma-2-2b-it


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.41it/s]


Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
2026-05-25 17:22:56 | Steering.data.loader                | INFO     | Loaded 700 composite samples from refusal_cast_responses (augmentation=1)
hook_name: blocks.14.hook_resid_pre
layer: 14
samples: 700
flow_steps: 20
vector_dir: /home/caotue/SAESteeringBench/Vector/Flow/gemma_refusal_response_subspace_l4_e200


In [8]:
vec = torch.load(vector_dir / "vector.pt", map_location="cpu")
meta = torch.load(vector_dir / "metadata.pt", map_location="cpu")

if not isinstance(vec, dict):
    raise TypeError(f"Expected vector.pt dict payload, got {type(vec).__name__}")
if not isinstance(meta, dict):
    raise TypeError(f"Expected metadata.pt dict payload, got {type(meta).__name__}")

flow_models = meta.get("flow_models")
if not isinstance(flow_models, dict):
    raise KeyError("metadata.pt missing 'flow_models' dictionary")

layer_key = int(layer)
if layer_key not in flow_models:
    raise KeyError(f"Layer {layer_key} not found in flow_models. Available: {sorted(flow_models.keys())}")

flow_payload = flow_models[layer_key]
flow_transport = build_flow_inference_fn(flow_payload, flow_steps=flow_steps)

target_acts = collect_last_token_activations(model, target_texts, hook_name, batch_size=batch_size, max_length=max_length)
contrast_acts = collect_last_token_activations(model, contrast_texts, hook_name, batch_size=batch_size, max_length=max_length)
flow_acts = flow_transport(contrast_acts).cpu()

n_eval = min(target_acts.shape[0], contrast_acts.shape[0], flow_acts.shape[0])
target_acts = target_acts[:n_eval]
contrast_acts = contrast_acts[:n_eval]
flow_acts = flow_acts[:n_eval]

print("target_acts:", tuple(target_acts.shape))
print("contrast_acts:", tuple(contrast_acts.shape))
print("flow_acts:", tuple(flow_acts.shape))

target_acts: (700, 2304)
contrast_acts: (700, 2304)
flow_acts: (700, 2304)


In [9]:
fd_contrast, fd_ratio_contrast = compute_fd(target_acts, contrast_acts, seed=42)
fd_flow, fd_ratio_flow = compute_fd(target_acts, flow_acts, seed=42)

fd1d_contrast = compute_fd_1d(target_acts, contrast_acts)
fd1d_flow = compute_fd_1d(target_acts, flow_acts)

summary = pd.DataFrame([
    {
        "pair": "target vs contrast",
        "fd": float(fd_contrast),
        "fd_ratio": float(fd_ratio_contrast),
        "fd_1d_mean": float(np.mean(fd1d_contrast["fd_1d"])),
        "fd_1d_median": float(np.median(fd1d_contrast["fd_1d"])),
    },
    {
        "pair": "target vs flow(contrast)",
        "fd": float(fd_flow),
        "fd_ratio": float(fd_ratio_flow),
        "fd_1d_mean": float(np.mean(fd1d_flow["fd_1d"])),
        "fd_1d_median": float(np.median(fd1d_flow["fd_1d"])),
    },
])

print("Lower is better for FD.")
display(summary)

top_k = 20
df_dims = pd.DataFrame({
    "dim": np.arange(len(fd1d_flow["fd_1d"])),
    "fd1d_contrast": fd1d_contrast["fd_1d"],
    "fd1d_flow": fd1d_flow["fd_1d"],
    "improvement": fd1d_contrast["fd_1d"] - fd1d_flow["fd_1d"],
}).sort_values("improvement", ascending=False).reset_index(drop=True)

display(df_dims.head(top_k))

Lower is better for FD.


,pair,fd,fd_ratio,fd_1d_mean,fd_1d_median
0,target vs contrast,28889.373242,0.619681,4.464171,1.719629
1,target vs flow(contrast),20308.102087,0.435612,4.613555,1.786396


,dim,fd1d_contrast,fd1d_flow,improvement
0,714,222.953276,30.796801,192.156476
1,1645,242.774050,58.578653,184.195397
2,2287,162.813598,3.443769,159.369830
3,1281,158.384815,2.974437,155.410378
4,1670,84.651091,0.306710,84.344381
5,2106,72.938142,6.336324,66.601818
6,1037,61.598555,7.615737,53.982818
7,2128,59.232859,6.436396,52.796463
8,2042,58.040171,13.983226,44.056945
9,986,43.525515,2.784133,40.741382
